# Module 3 — Take-Home Exercises: Deployment & Inference

These exercises reinforce what you learned about deploying fine-tuned models,
running inference, and understanding generation parameters.

**Exercise 1-2:** T4 GPU required (Colab free tier)
**Exercise 3:** No GPU needed (analysis only)

---

## Exercise 1: Generation Parameter Exploration (T4 GPU Required)

In class, we used `temperature=0.7` and `top_p=0.9` for generation.
These parameters control how "creative" vs "deterministic" the model is.

**Tasks:**

1. Run the SAME question with 4 different temperature settings: 0.1, 0.5, 0.7, 1.0
2. Run the SAME question 3 times at temperature=1.0 — how much do outputs vary?
3. Compare temperature=0.1 vs temperature=1.0 for a medical question.
   Which would you choose for a healthcare chatbot? Why?
4. Test `top_p=0.5` vs `top_p=0.95` — what changes?

In [ ]:
!pip install -q transformers torch bitsandbytes accelerate peft

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
ADAPTER_REPO = "jeev1992/healthcare-assistant-lora-v2"  # v2 adapter from class

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb_config, device_map="auto")
model = PeftModel.from_pretrained(base_model, ADAPTER_REPO)

SYSTEM_PROMPT = "You are a helpful healthcare assistant. Provide accurate medical information and always recommend consulting a healthcare professional."

def generate(prompt, temperature=0.7, top_p=0.9, max_new_tokens=300):
    messages = [{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=top_p,
            do_sample=True if temperature > 0 else False
        )
    return tokenizer.decode(outputs[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
# Task 1: Temperature comparison
question = "What are the early warning signs of a stroke?"

temperatures = [0.1, 0.5, 0.7, 1.0]

for temp in temperatures:
    print(f"\n{'='*60}")
    print(f"Temperature = {temp}")
    print(f"{'='*60}")
    # TODO: Generate and print response
    # response = generate(question, temperature=temp)
    # print(response)

In [ ]:
# Task 2: Repeatability test at temperature=1.0
question = "How does metformin work for diabetes management?"

print("Running same question 3 times at temperature=1.0:\n")
for i in range(3):
    print(f"\n--- Run {i+1} ---")
    # TODO: Generate and print
    # response = generate(question, temperature=1.0)
    # print(response)

# Question: How much did the 3 outputs differ?
# Would this variability be acceptable for a medical chatbot?

In [ ]:
# Task 4: top_p comparison
question = "What vaccinations are recommended for adults over 65?"

# TODO: Compare top_p=0.5 vs top_p=0.95 (keep temperature=0.7)
# print("--- top_p=0.5 ---")
# print(generate(question, temperature=0.7, top_p=0.5))
# print("\n--- top_p=0.95 ---")
# print(generate(question, temperature=0.7, top_p=0.95))

**Your Findings (fill in):**

1. At temperature=0.1, the output is: (deterministic / creative / random)
2. At temperature=1.0 across 3 runs, variability was: (none / some / high)
3. For a healthcare chatbot, I'd choose temperature=___ because...
4. top_p=0.5 vs top_p=0.95: the difference I noticed was...

---

## Exercise 2: Adapter Enable/Disable Comparison (T4 GPU Required)

In class, we used `disable_adapter_layers()` to compare base vs fine-tuned
using the same loaded model. This exercise makes you do it hands-on.

**Tasks:**

1. For each of the 3 questions below:
   - Generate with adapters DISABLED (base model behavior)
   - Generate with adapters ENABLED (fine-tuned behavior)
2. For each pair, note:
   - Which has better medical accuracy?
   - Which includes safety disclaimers?
   - Which is more detailed/helpful?

In [ ]:
test_questions = [
    "How is pneumonia typically diagnosed?",
    "Explain the difference between Type 1 and Type 2 diabetes.",
    "What are the common side effects of ibuprofen?",
]

for q in test_questions:
    print(f"\n{'='*70}")
    print(f"Q: {q}")
    
    # TODO: Disable adapters, generate base output
    # model.disable_adapter_layers()
    # base_output = generate(q, temperature=0.7)
    
    # TODO: Enable adapters, generate fine-tuned output
    # model.enable_adapter_layers()
    # ft_output = generate(q, temperature=0.7)
    
    # print(f"\n--- BASE MODEL ---")
    # print(base_output)
    # print(f"\n--- FINE-TUNED ---")
    # print(ft_output)

**Your Comparison (fill in for each question):**

| Question | Better Accuracy | Has Safety Disclaimer | More Detailed |
|---|---|---|---|
| Pneumonia diagnosed | base / ft | base / ft / both | base / ft |
| Type 1 vs Type 2 | base / ft  | base / ft / both | base / ft |
| Ibuprofen side effects | base / ft | base / ft / both | base / ft |

---

## Exercise 3: Build an Inference Cost Calculator (No GPU)

If you were to deploy this model in production, you'd need to estimate costs.

**Scenario:** Your hospital wants to serve 1,000 patient queries per day
using the fine-tuned model.

**Tasks:**

1. Estimate the average input + output tokens per query
   (use the benchmark results to measure actual token counts)
2. Calculate daily token throughput (1,000 queries × avg tokens)
3. Compare the cost of 3 deployment options:
   - **HF Inference Endpoints** (dedicated GPU): ~$1.30/hr for a T4
   - **AWS SageMaker** (ml.g5.xlarge): ~$1.41/hr
   - **Self-hosted** (RunPod A10G): ~$0.36/hr
4. Which option is most cost-effective? What factors besides cost matter?

In [ ]:
import json

# Load benchmark results to estimate token counts
with open("../module_2_colab_finetuning/results/benchmark_results_v2.json") as f:
    results = json.load(f)

# Rough estimate: 1 token ≈ 4 characters for English text
CHARS_PER_TOKEN = 4

# TODO: Calculate average input and output token counts
input_tokens = []
output_tokens = []

for r in results:
    # Input = system prompt (~50 tokens) + user question
    input_len = 50 + len(r["prompt"]) // CHARS_PER_TOKEN
    # Output = fine-tuned response
    output_len = len(r["finetuned_output"]) // CHARS_PER_TOKEN
    input_tokens.append(input_len)
    output_tokens.append(output_len)

avg_input = sum(input_tokens) / len(input_tokens)
avg_output = sum(output_tokens) / len(output_tokens)
avg_total = avg_input + avg_output

print(f"Average tokens per query: {avg_total:.0f} ({avg_input:.0f} input + {avg_output:.0f} output)")

In [ ]:
# TODO: Calculate monthly costs for 3 deployment options

QUERIES_PER_DAY = 1000
DAYS_PER_MONTH = 30

# Assume the model can handle ~10 queries/sec on T4 (short responses)
# That means 1000 queries takes ~100 seconds = ~2 minutes
# But you need the server running 24/7 for availability

deployment_options = {
    "HF Inference Endpoints (T4)": 1.30,   # $/hr
    "AWS SageMaker (g5.xlarge)":   1.41,   # $/hr
    "RunPod (A10G)": 0.36,                  # $/hr
}

print(f"{'Option':<35} {'$/hr':>6} {'$/day':>8} {'$/month':>10}")
print("-" * 65)

for name, hourly_rate in deployment_options.items():
    daily = None    # TODO: hourly_rate * 24
    monthly = None  # TODO: daily * 30
    print(f"{name:<35} ${hourly_rate:>5.2f} ${daily:>7.2f} ${monthly:>9.2f}")

**Your Analysis (fill in):**

1. Most cost-effective option: ___ at $___/month
2. Factors besides cost that matter for a hospital deployment:
   - ...
   - ...
   - ...
3. Would you run the server 24/7 or use auto-scaling? Why?
4. At what query volume does self-hosting become cheaper than a managed service?